# FarmGenius - Disease Detection Model Training
This notebook trains a TFLite model using the PlantVillage dataset on Kaggle, optimized for Indian crops.
## 1. Install Dependencies

In [ ]:
!pip install -q tensorflow kaggle Pillow
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## 2. Download PlantVillage Dataset

In [ ]:
!kaggle datasets download -d emmarex/plantdisease
!unzip -q plantdisease.zip -d dataset

## 3. Filter to Indian Crops Only
Keep only Tomato, Potato, Rice, Wheat, Cotton, Chickpea, Maize.

In [ ]:
import os
import shutil

dataset_dir = 'dataset/PlantVillage'
indian_crops = ['Tomato', 'Potato', 'Rice', 'Wheat', 'Cotton', 'Chickpea', 'Maize', 'Corn']

if os.path.exists(dataset_dir):
    for folder in os.listdir(dataset_dir):
        if not any(crop.lower() in folder.lower() for crop in indian_crops):
            shutil.rmtree(os.path.join(dataset_dir, folder))
            print(f"Removed non-target crop: {folder}")

## 4. Data Augmentation
We use random rotation, flip, and brightness to simulate field photos.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)
print(f"Found {num_classes} classes")

## 5. MobileNetV2 Transfer Learning

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

## 6. Fine-Tuning

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

## 7. Evaluate and Save Labels

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

Y_pred = model.predict(val_generator)
y_pred = np.argmax(Y_pred, axis=1)
print("Classification Report")
print(classification_report(val_generator.classes, y_pred, target_names=class_names, zero_division=0))

with open('labels_indian_crops.txt', 'w') as f:
    for c in class_names:
        f.write(f"{c}\n")
    f.write("Unknown — please retake photo\n")

## 8. Export to TFLite with INT8 Quantization

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_data_gen():
    for i in range(100):
        batch = next(train_generator)
        yield [batch[0]]

converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_quant = converter.convert()

with open('disease_model_quant.tflite', 'wb') as f:
    f.write(tflite_model_quant)

## 9. Verify Model Size

In [ ]:
import os
size_mb = os.path.getsize('disease_model_quant.tflite') / (1024 * 1024)
print(f"Model size: {size_mb:.2f} MB")
assert size_mb < 15, "Model size exceeds 15MB limit!"